# 12-04 SQL 高频面试题

**工具**: DuckDB (内存数据库，无需安装 MySQL)

**高频考点**: 窗口函数、留存分析、漏斗转化、连续登录、TopN per group、CTE

---

In [ ]:
# 环境准备: DuckDB
# pip install duckdb
import duckdb

con = duckdb.connect()

# 创建模拟数据: B站广告投放数据
con.execute("""
CREATE TABLE ad_clicks AS
SELECT * FROM (VALUES
    (1, 1001, '2024-01-01', 'CPC', 1000, 50, 5, 2.0),
    (2, 1001, '2024-01-02', 'CPC', 1200, 60, 8, 2.5),
    (3, 1001, '2024-01-03', 'CPC', 800, 40, 3, 1.8),
    (4, 1002, '2024-01-01', 'CPM', 5000, 200, 15, 3.0),
    (5, 1002, '2024-01-02', 'CPM', 4500, 180, 12, 2.8),
    (6, 1002, '2024-01-03', 'CPM', 5200, 210, 18, 3.2),
    (7, 1003, '2024-01-01', 'OCPC', 3000, 150, 20, 5.0),
    (8, 1003, '2024-01-02', 'OCPC', 3200, 160, 22, 5.5),
    (9, 1003, '2024-01-03', 'OCPC', 2800, 140, 18, 4.8),
    (10, 1001, '2024-01-04', 'CPC', 1100, 55, 6, 2.2),
    (11, 1002, '2024-01-04', 'CPM', 4800, 190, 14, 2.9),
    (12, 1003, '2024-01-04', 'OCPC', 3100, 155, 21, 5.2)
) AS t(id, advertiser_id, dt, bid_type, impressions, clicks, conversions, cost_yuan)
""")

# 创建用户行为数据
con.execute("""
CREATE TABLE user_actions AS
SELECT * FROM (VALUES
    (101, '2024-01-01', 'view'),
    (101, '2024-01-01', 'click'),
    (101, '2024-01-02', 'view'),
    (101, '2024-01-02', 'click'),
    (101, '2024-01-02', 'convert'),
    (101, '2024-01-03', 'view'),
    (101, '2024-01-05', 'view'),
    (102, '2024-01-01', 'view'),
    (102, '2024-01-01', 'click'),
    (102, '2024-01-03', 'view'),
    (102, '2024-01-04', 'view'),
    (102, '2024-01-04', 'click'),
    (103, '2024-01-02', 'view'),
    (103, '2024-01-02', 'click'),
    (103, '2024-01-02', 'convert'),
    (103, '2024-01-03', 'view'),
    (103, '2024-01-03', 'click'),
    (103, '2024-01-04', 'view'),
    (103, '2024-01-05', 'view'),
    (103, '2024-01-05', 'click'),
    (104, '2024-01-01', 'view'),
    (104, '2024-01-02', 'view'),
    (104, '2024-01-03', 'view'),
    (104, '2024-01-04', 'view'),
    (104, '2024-01-05', 'view'),
    (105, '2024-01-03', 'view'),
    (105, '2024-01-04', 'view'),
    (105, '2024-01-04', 'click')
) AS t(user_id, dt, action)
""")

print('数据表创建成功!')
print('\nad_clicks:')
print(con.execute('SELECT * FROM ad_clicks LIMIT 5').fetchdf())
print('\nuser_actions:')
print(con.execute('SELECT * FROM user_actions LIMIT 5').fetchdf())

In [ ]:
# 题目1: 窗口函数 — 计算每日 CTR 及其 3日移动平均
print("""题目: 计算每个广告主每天的 CTR (点击率=clicks/impressions)，
并计算 3日移动平均 CTR""")
print()

result = con.execute("""
SELECT 
    advertiser_id,
    dt,
    clicks,
    impressions,
    ROUND(clicks * 1.0 / impressions, 4) AS ctr,
    ROUND(AVG(clicks * 1.0 / impressions) OVER (
        PARTITION BY advertiser_id 
        ORDER BY dt 
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 4) AS ctr_3d_avg
FROM ad_clicks
ORDER BY advertiser_id, dt
""").fetchdf()

print(result)
print()
print("""知识点:
- AVG() OVER (PARTITION BY ... ORDER BY ... ROWS BETWEEN N PRECEDING AND CURRENT ROW)
- ROWS BETWEEN: 物理行窗口
- RANGE BETWEEN: 逻辑值窗口 (基于 ORDER BY 的值范围)""")

In [ ]:
# 题目2: 窗口函数 — 每个出价类型的 Top 1 广告主 (按转化量)
print("""题目: 每种出价类型 (CPC/CPM/OCPC) 中，总转化量最高的广告主是谁？\n""")

result = con.execute("""
WITH ranked AS (
    SELECT
        bid_type,
        advertiser_id,
        SUM(conversions) AS total_conversions,
        SUM(cost_yuan) AS total_cost,
        ROW_NUMBER() OVER (
            PARTITION BY bid_type 
            ORDER BY SUM(conversions) DESC
        ) AS rk
    FROM ad_clicks
    GROUP BY bid_type, advertiser_id
)
SELECT bid_type, advertiser_id, total_conversions, total_cost
FROM ranked
WHERE rk = 1
ORDER BY bid_type
""").fetchdf()

print(result)
print()
print("""知识点:
- ROW_NUMBER() vs RANK() vs DENSE_RANK():
  ROW_NUMBER: 1,2,3,4 (不并列)
  RANK:       1,2,2,4 (并列跳号)
  DENSE_RANK: 1,2,2,3 (并列不跳号)
- TopN per group 经典套路: CTE + ROW_NUMBER + WHERE rk <= N""")

In [ ]:
# 题目3: 漏斗分析 — view → click → convert 转化率
print("""题目: 计算广告漏斗每一步的转化率 (view→click→convert)\n""")

result = con.execute("""
WITH funnel AS (
    SELECT
        COUNT(DISTINCT CASE WHEN action = 'view' THEN user_id END) AS view_users,
        COUNT(DISTINCT CASE WHEN action = 'click' THEN user_id END) AS click_users,
        COUNT(DISTINCT CASE WHEN action = 'convert' THEN user_id END) AS convert_users
    FROM user_actions
)
SELECT
    view_users,
    click_users,
    convert_users,
    ROUND(click_users * 100.0 / view_users, 1) AS view_to_click_pct,
    ROUND(convert_users * 100.0 / click_users, 1) AS click_to_convert_pct,
    ROUND(convert_users * 100.0 / view_users, 1) AS overall_convert_pct
FROM funnel
""").fetchdf()

print(result)
print()
print("""知识点:
- COUNT(DISTINCT CASE WHEN ... THEN user_id END) 做条件去重计数
- 漏斗分析核心: 每步的转化率 = 当前步人数 / 上一步人数
- 注意: 这里是简化版，没考虑顺序 (严格漏斗需要保证 view 在 click 之前)
- 严格漏斗可用窗口函数 + 自连接实现""")

In [ ]:
# 题目4: 连续登录天数 (经典高频题)
print("""题目: 找出连续活跃 >= 3 天的用户\n""")

result = con.execute("""
WITH daily_active AS (
    -- 先去重: 每个用户每天只算一次
    SELECT DISTINCT user_id, CAST(dt AS DATE) AS dt
    FROM user_actions
),
with_group AS (
    -- 核心技巧: dt - ROW_NUMBER = 同一个值说明日期连续
    SELECT
        user_id,
        dt,
        dt - INTERVAL (ROW_NUMBER() OVER (
            PARTITION BY user_id ORDER BY dt
        )) DAY AS grp
    FROM daily_active
),
consecutive AS (
    SELECT
        user_id,
        grp,
        MIN(dt) AS start_date,
        MAX(dt) AS end_date,
        COUNT(*) AS consecutive_days
    FROM with_group
    GROUP BY user_id, grp
)
SELECT user_id, start_date, end_date, consecutive_days
FROM consecutive
WHERE consecutive_days >= 3
ORDER BY user_id, start_date
""").fetchdf()

print(result)
print()
print("""知识点 (连续问题万能解法):
1. 去重: 每个用户每天只保留一条
2. 分组: dt - ROW_NUMBER() → 连续日期会得到相同的值
   例: 1月1日-row1=12月31日, 1月2日-row2=12月31日, 1月3日-row3=12月31日 → 同组
3. 聚合: GROUP BY (user_id, grp) 统计每段连续天数
4. 过滤: HAVING/WHERE consecutive_days >= N""")

In [ ]:
# 题目5: 留存率分析
print("""题目: 计算 2024-01-01 新用户的次日、3日、7日留存率\n""")

result = con.execute("""
WITH first_day AS (
    -- 每个用户的首次活跃日期
    SELECT user_id, MIN(CAST(dt AS DATE)) AS first_dt
    FROM user_actions
    GROUP BY user_id
),
new_users AS (
    -- 2024-01-01 的新用户
    SELECT user_id
    FROM first_day
    WHERE first_dt = '2024-01-01'
),
retention AS (
    SELECT
        n.user_id,
        CAST(a.dt AS DATE) - DATE '2024-01-01' AS day_diff
    FROM new_users n
    LEFT JOIN (
        SELECT DISTINCT user_id, dt FROM user_actions
    ) a ON n.user_id = a.user_id
)
SELECT
    COUNT(DISTINCT user_id) AS day0_users,
    COUNT(DISTINCT CASE WHEN day_diff = 1 THEN user_id END) AS day1_retained,
    COUNT(DISTINCT CASE WHEN day_diff = 2 THEN user_id END) AS day2_retained,
    COUNT(DISTINCT CASE WHEN day_diff = 3 THEN user_id END) AS day3_retained,
    ROUND(COUNT(DISTINCT CASE WHEN day_diff = 1 THEN user_id END) * 100.0 /
          COUNT(DISTINCT user_id), 1) AS day1_retention_pct,
    ROUND(COUNT(DISTINCT CASE WHEN day_diff = 2 THEN user_id END) * 100.0 /
          COUNT(DISTINCT user_id), 1) AS day2_retention_pct,
    ROUND(COUNT(DISTINCT CASE WHEN day_diff = 3 THEN user_id END) * 100.0 /
          COUNT(DISTINCT user_id), 1) AS day3_retention_pct
FROM retention
""").fetchdf()

print(result.to_string(index=False))
print()
print("""知识点:
- 留存率 = 第N天仍活跃的用户数 / 第0天新增用户数
- 关键步骤: 1) 找新用户首日 2) LEFT JOIN 后续活跃 3) 按天差聚合
- day_diff = DATEDIFF(activity_date, first_date)
- 变体: 7日留存、30日留存、自然周留存""")

In [ ]:
# 题目6: 累计求和 + LAG/LEAD
print("""题目: 计算每个广告主的累计花费，以及与前一天的花费环比\n""")

result = con.execute("""
SELECT
    advertiser_id,
    dt,
    cost_yuan,
    SUM(cost_yuan) OVER (
        PARTITION BY advertiser_id 
        ORDER BY dt
    ) AS cumulative_cost,
    LAG(cost_yuan) OVER (
        PARTITION BY advertiser_id 
        ORDER BY dt
    ) AS prev_day_cost,
    ROUND(
        (cost_yuan - LAG(cost_yuan) OVER (PARTITION BY advertiser_id ORDER BY dt)) 
        / LAG(cost_yuan) OVER (PARTITION BY advertiser_id ORDER BY dt) * 100
    , 1) AS cost_change_pct
FROM ad_clicks
ORDER BY advertiser_id, dt
""").fetchdf()

print(result)
print()
print("""知识点:
- SUM() OVER (ORDER BY dt): 默认从第一行累加到当前行
- LAG(col, N, default): 取前 N 行的值 (默认 N=1)
- LEAD(col, N, default): 取后 N 行的值
- 环比 = (当前值 - 上期值) / 上期值 * 100%""")

In [ ]:
# 题目7: CTE + 去重 — 每个用户最近一次转化记录
print("""题目: 找到每个有转化的用户最近一次转化的日期和之前的行为序列\n""")

result = con.execute("""
WITH convert_users AS (
    -- 找到有转化行为的用户和最近转化日期
    SELECT 
        user_id,
        MAX(CAST(dt AS DATE)) AS last_convert_dt
    FROM user_actions
    WHERE action = 'convert'
    GROUP BY user_id
),
user_journey AS (
    -- 拉取这些用户在转化日当天及之前的行为
    SELECT
        a.user_id,
        a.dt,
        a.action,
        c.last_convert_dt,
        ROW_NUMBER() OVER (PARTITION BY a.user_id ORDER BY a.dt, a.action) AS step
    FROM user_actions a
    JOIN convert_users c ON a.user_id = c.user_id
    WHERE CAST(a.dt AS DATE) <= c.last_convert_dt
)
SELECT user_id, step, dt, action, last_convert_dt
FROM user_journey
ORDER BY user_id, step
""").fetchdf()

print(result)
print()
print("""知识点:
- CTE (Common Table Expression): WITH ... AS (...) 让复杂查询分步可读
- 多个 CTE 可以链式引用
- 用户行为序列分析是广告场景常见需求""")

## SQL 面试速查卡片

| 题型 | 核心技巧 | 关键函数 |
|------|---------|----------|
| 窗口函数 | OVER (PARTITION BY ... ORDER BY ...) | ROW_NUMBER, RANK, DENSE_RANK, LAG, LEAD |
| 移动平均 | ROWS BETWEEN N PRECEDING AND CURRENT ROW | AVG/SUM OVER |
| TopN per group | CTE + ROW_NUMBER + WHERE rk <= N | ROW_NUMBER |
| 连续N天 | dt - ROW_NUMBER() 分组法 | ROW_NUMBER, GROUP BY |
| 留存分析 | 新用户首日 + LEFT JOIN + DATEDIFF 分桶 | MIN, DATEDIFF, CASE WHEN |
| 漏斗分析 | COUNT(DISTINCT CASE WHEN action=x THEN user_id) | COUNT DISTINCT + CASE |
| 累计求和 | SUM() OVER (ORDER BY dt) | SUM OVER |
| 环比/同比 | LAG() 取上期值再计算变化率 | LAG, LEAD |
| 去重 | DISTINCT / GROUP BY / ROW_NUMBER 取 rk=1 | DISTINCT |
| CTE 递归 | WITH RECURSIVE cte AS (base UNION ALL recursive) | WITH RECURSIVE |